In [0]:
# !pip install statsmodels
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [0]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1. SQL Otimizado (Calcula tudo no Spark primeiro!)
query = """
SELECT 
    t.date_time, 
    SUM(f.energy_spent) AS energy_spent,
    -- Como a meteorologia é igual para a mesma hora, usamos MAX para não duplicar linhas
    MAX(w.temp_bin_10) AS temp_bin_10
FROM smart_meters.gold.fact_consumption f
LEFT JOIN smart_meters.gold.dim_time t ON f.dim_time_id = t.id
JOIN smart_meters.gold.dim_weather w ON f.sk_weather = w.sk_weather
GROUP BY t.date_time
ORDER BY t.date_time
"""

# Agora sim, o toPandas() vai trazer apenas ~2200 linhas (muito leve!)
df = spark.sql(query).toPandas()

# 2. Preparação do Índice e Preenchimento de Falhas
df['date_time'] = pd.to_datetime(df['date_time'])
df.set_index('date_time', inplace=True)
df = df.asfreq('H').fillna(method='ffill')

# 3. Transformar o Texto em Dummy Variables e FORÇAR PARA INTEIROS (dtype=int)
df_encoded = pd.get_dummies(df, columns=['temp_bin_10'], drop_first=True, dtype=int)

# (Opcional, mas seguro) Se ainda assim a coluna energy_spent for vista como object/boolean
df_encoded['energy_spent'] = df_encoded['energy_spent'].astype(float)

# 4. Separar Variáveis
y = df_encoded['energy_spent']
exog = df_encoded.drop(columns=['energy_spent'])

# 5. Treinar o SARIMAX
print("A treinar o modelo... Isto pode demorar 1 ou 2 minutos.")
model = SARIMAX(y, 
                exog=exog, 
                order=(1, 1, 1), 
                seasonal_order=(1, 1, 1, 24),
                enforce_stationarity=False,  # Deixa o modelo mais rápido e tolerante
                enforce_invertibility=False) # Evita erros matemáticos de convergência

results = model.fit(disp=False) # disp=False esconde o spam de texto do treino

print("Modelo treinado com sucesso!")
print(results.summary())

In [0]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Fazer previsões in-sample
y_pred = results.fittedvalues

# Calcular métricas
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mape = np.mean(np.abs((y - y_pred) / y)) * 100

# Média do consumo real
mean_energy = y.mean()

print("="*50)
print("MÉTRICAS DE PRECISÃO DO MODELO")
print("="*50)
print(f"MAE (Mean Absolute Error):      {mae:.2f} kWh")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} kWh")
print(f"MAPE (Mean Absolute % Error):   {mape:.2f}%")
print(f"\nConsumo Médio Real:             {mean_energy:.2f} kWh")
print(f"Erro Relativo (RMSE/Média):     {(rmse/mean_energy)*100:.2f}%")
print("="*50)

# Resumo dos resíduos
residuals = results.resid
print(f"\nRESÍDUOS:")
print(f"Média:      {residuals.mean():.2f} (deve estar perto de 0)")
print(f"Std Dev:    {residuals.std():.2f}")
print(f"Min:        {residuals.min():.2f}")
print(f"Max:        {residuals.max():.2f}")

In [0]:
import matplotlib.pyplot as plt

# Série Temporal: Real vs Previsto (primeiras 168h = 1 semana)
fig, ax = plt.subplots(figsize=(16, 6))

week_data = y.iloc[:168]
week_pred = y_pred.iloc[:168]

ax.plot(week_data.index, week_data, label='Real', linewidth=2, alpha=0.8)
ax.plot(week_pred.index, week_pred, label='Previsto', linewidth=2, alpha=0.8)
ax.set_title('Primeira Semana: Real vs Previsto', fontsize=14, fontweight='bold')
ax.set_xlabel('Data', fontsize=12)
ax.set_ylabel('Consumo (kWh)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
display(plt.show())

In [0]:
import re

# Configuration
VOLUME_PATH = "/Volumes/smart_meters/bronze/tad_volume"

# ==========================================
# 1. INGESTÃO DO DATASET DE CONSUMO (hhblock)
# ==========================================
consumption_path = f"{VOLUME_PATH}/hhblock_dataset/*.csv"
table_consumption_path = f"smart_meters.bronze.consumption"

try:
    df_consumption = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(consumption_path)

    # Gravar na Bronze
    df_consumption.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_consumption_path)
    
    print(f"✅ Tabela de consumo guardada em {table_consumption_path}! (Linhas lidas: {df_consumption.count()})")
except Exception as e:
    print(f"❌ Erro ao carregar o consumo: {e}")


# ==========================================
# 2. INGESTÃO DAS TABELAS DE REFERÊNCIA
# ==========================================
reference_files = {
    "households": ("informations_households.csv", ","),
    "weather": ("weather_hourly_darksky.csv", ","),
    "tariffs": ("Tariffs.csv", ";"),
    "holidays": ("uk_bank_holidays.csv", ",")
}

for table_name, (file_name, dlmtr) in reference_files.items():
    file_path = f"{VOLUME_PATH}/{file_name}"
    
    try:
        df_spark = spark.read.format("csv") \
            .option("header", "true") \
            .option("sep", dlmtr) \
            .option("inferSchema", "true") \
            .load(file_path)
        
        # Limpeza de nomes de colunas
        for col_name in df_spark.columns:
            clean_name = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name.strip())
            df_spark = df_spark.withColumnRenamed(col_name, clean_name)
        
        table_path = f"smart_meters.bronze.{table_name}"
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)
        
        print(f"✅ Tabela '{table_path}' criada com colunas higienizadas!")
        
    except Exception as e:
        print(f"❌ Erro ao carregar {file_name}: {e}")